# **CHAT WITH MULTIPLE PDFS :**

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 52.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 75.4 MB/s eta 0:00:00:00:0100:01


In [3]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 26.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.66
    Uninstalling langchain-core-0.3.66:
      Successfully uninstalled langchain-core-0.3.66
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.8
    Uninstalling langchain-text-splitters-0.3.8:
      Successfully uninstalled langchain-text-splitters-0.3.8
  Attempting uninstall: l

In [4]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.0 MB/s eta 0:00:00a 0:00:01


In [5]:
!pip install langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.5 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.6.15
    Uninstalling google-ai-generativelanguage-0.6.15:
      Successfully uninstalled google-ai-generativelanguage-0.6.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.7.0 which is incompatible.


In [6]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 48.8 MB/s eta 0:00:00:00:0100:01


In [7]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 2.8 MB/s eta 0:00:0000:01


In [8]:
!pip install --upgrade google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.4 MB/s eta 0:00:0000:010:01
  Attempting uninstall: google-ai-generativelanguage
    Found existing installation: google-ai-generativelanguage 0.7.0
    Uninstalling google-ai-generativelanguage-0.7.0:
      Successfully uninstalled google-ai-generativelanguage-0.7.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-google-genai 2.1.12 requires google-ai-generativelanguage<1,>=0.7, but you have google-ai-generativelanguage 0.6.15 which is incompatible.


## Import libraries : 

In [9]:
import streamlit as st
from PyPDF2 import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os
#from langchain_google_genai import GoogleGenerativeAIEmbeddings
#import google.generativeai as genai
from langchain.vectorstores import FAISS
#from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
from langchain.chains.question_answering import load_qa_chain

In [10]:
import gradio as gr
from langchain_groq import ChatGroq

## Google api key :

In [11]:
import os
os.environ["GROQ_API_KEY"]="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"

In [12]:
#genai.configure(api_key=os.environ["GROQ_API_KEY"])

## Loading pdf : 

In [13]:
def get_pdf_text(pdf_docs):
    text=""
    for pdf in pdf_docs:
        pdf_reader=PdfReader(pdf)
        for page in pdf_reader.pages:
            text+=page.extract_text()
    return text

## Convert texts into chunks :

In [14]:
def get_text_chunks(text):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=50
    )
    chunks=text_splitter.split_text(text)
    return chunks

## Convert chunks into vectors :

In [15]:
from langchain.embeddings import HuggingFaceEmbeddings

In [16]:
def get_vector_store(text_chunks):
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_store=FAISS.from_texts(text_chunks,embedding=embeddings)
    vector_store.save_local("faiss_index")

## Conversational Chain :

In [22]:
from langchain.prompts import PromptTemplate
from langchain.chains.question_answering import load_qa_chain
from langchain_google_genai import ChatGoogleGenerativeAI

def get_conversational_chain():
    prompt_template = """
    Answer the question as detailed as possible from the provided context. 
    Make sure to provide all the details. 
    If the answer is not in the provided context, just say: 
    "Answer is not available in the context". 
    Do not make up an answer. 

    Context:
    {context}

    Question: 
    {question}

    Answer:
    """
    
    model=ChatGroq(model="gemma2-9b-it")    
    prompt = PromptTemplate(
        template=prompt_template,
        input_variables=["context", "question"]
    )
    
    chain = load_qa_chain(model, prompt=prompt, chain_type="stuff")
    return chain


## User input Pdf :

In [23]:
def user_input(user_question):
    embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    new_db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
    docs=new_db.similarity_search(user_question)
    chain=get_conversational_chain()

    response=chain(
        {"input_documents":docs,"question":user_question},
        return_only_outputs=True
    )
    print(response)
    st.write("Reply : ",response["output_text"])

## Main function : 

In [24]:
def process_pdfs(pdf_files, question):
    # Step 1: Extract & process PDFs
    raw_text = get_pdf_text(pdf_files)
    chunks = get_text_chunks(raw_text)
    get_vector_store(chunks)

    # Step 2: Answer the question
    if question:
        answer = user_input(question)
    else:
        answer = "Ask a question to get an answer!"
    return answer


with gr.Blocks() as demo:
    gr.Markdown("# Chat with PDF using Gemini")
    
    with gr.Row():
        pdf_input = gr.File(
            file_types=[".pdf"],     # ✅ valid
            file_count="multiple",   # allow multiple PDFs
            label="Upload PDF(s)"    # this will show as description
        )

        question_input = gr.Textbox(lines=1, placeholder="Ask a question about the PDFs...", label="Your Question")
    
    output = gr.Textbox(lines=5, label="Answer")
    submit_btn = gr.Button("Submit & Process")
    
    submit_btn.click(
        fn=process_pdfs,
        inputs=[pdf_input, question_input],
        outputs=output
    )

demo.launch()


* Running on local URL:  http://127.0.0.1:7862
It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://9b2c9c3b1d5897bdbd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2025-09-23 06:43:15.574 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'AnyIO worker thread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-23 06:43:15.696 
  command:

    streamlit run /usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-09-23 06:43:15.697 Thread 'AnyIO worker thread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-23 06:43:15.698 Thread 'AnyIO worker thread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-23 06:43:15.698 Thread 'AnyIO worker thread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-23 06:43:15.699 Thread 'AnyIO worker thread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-23 06:43:15.701 Thread 'AnyIO worker thread': missing ScriptRunContext! This warning can be ignored when running in bare

{'output_text': 'Pinecone is a fully managed vector database designed for similarity search and retrieval-augmented generation (RAG) applications. It abstracts infrastructure management and focuses on high-dimensional vector indexing and querying.  \n'}
